# 🎬 Mi Mente — motor de IA GRATIS

Este cuaderno convierte una computadora gratis de Google en el motor de tu app:
- **Video con IA** (LTX-Video, open source): de texto a video y de imagen a video
- **Subtítulos** (Whisper), **música** (MusicGen) y **voz** (MMS)

### Cómo usarlo (2 minutos)
1. Arriba: **Entorno de ejecución → Cambiar tipo de entorno → GPU T4 → Guardar**.
2. **Entorno de ejecución → Ejecutar todo** (o dale ▶ a cada celda en orden).
3. Al final aparece un link `https://….trycloudflare.com`. Cópialo.
4. En la app: **Ajustes → Link de tu Colab** → pégalo → Guardar.

Deja esta pestaña abierta mientras usas la app. El link cambia cada vez que lo vuelves a arrancar.
La primera toma tarda más porque baja los modelos (~10 GB). Google da unas horas gratis al día.

In [ ]:
# 1) Instalar (1–3 min)
!pip -q install -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg fastapi uvicorn faster-whisper scipy
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NINGUNA — cambia el entorno a GPU T4')

In [ ]:
%%writefile /content/mimente_server.py
# Mi Mente — motor GRATIS en Google Colab (modelos open source).
# Video: LTX-Video (texto→video e imagen→video) · Subtítulos: Whisper · Música: MusicGen · Voz: MMS-TTS
import os, io, re, json, uuid, time, queue, base64, threading, traceback
import torch
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, JSONResponse
from pydantic import BaseModel

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

OUT = "/content/mimente_out"
os.makedirs(OUT, exist_ok=True)
JOBS = {}
Q = queue.Queue()
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
_models = {}


def data_url_bytes(u):
    return base64.b64decode(u.split(",", 1)[1] if "," in u else u)


# ---------- Modelos (se cargan la primera vez que se usan) ----------
def ltx():
    if "ltx" not in _models:
        from diffusers import LTXPipeline, LTXImageToVideoPipeline
        t2v = LTXPipeline.from_pretrained("Lightricks/LTX-Video", torch_dtype=DTYPE)
        i2v = LTXImageToVideoPipeline(**t2v.components)
        for p in (t2v, i2v):
            p.enable_model_cpu_offload()
        t2v.vae.enable_tiling()
        _models["ltx"] = (t2v, i2v)
    return _models["ltx"]


def whisper():
    if "whisper" not in _models:
        from faster_whisper import WhisperModel
        _models["whisper"] = WhisperModel("small", device="cuda" if torch.cuda.is_available() else "cpu",
                                          compute_type="float16" if torch.cuda.is_available() else "int8")
    return _models["whisper"]


def musicgen():
    if "music" not in _models:
        from transformers import AutoProcessor, MusicgenForConditionalGeneration
        proc = AutoProcessor.from_pretrained("facebook/musicgen-small")
        model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small").to("cuda" if torch.cuda.is_available() else "cpu")
        _models["music"] = (proc, model)
    return _models["music"]


def tts(lang):
    key = "tts_" + lang
    if key not in _models:
        from transformers import VitsModel, AutoTokenizer
        name = "facebook/mms-tts-spa" if lang == "spa" else "facebook/mms-tts-eng"
        _models[key] = (AutoTokenizer.from_pretrained(name), VitsModel.from_pretrained(name).to("cuda" if torch.cuda.is_available() else "cpu"))
    return _models[key]


def save_wav(path, audio, rate):
    import numpy as np
    import scipy.io.wavfile
    a = np.asarray(audio, dtype="float32").squeeze()
    a = a / max(1e-6, float(abs(a).max())) * 0.9
    scipy.io.wavfile.write(path, rate, (a * 32767).astype("int16"))


# ---------- Trabajos ----------
def run_video(job, p):
    from diffusers.utils import export_to_video
    from PIL import Image
    t2v, i2v = ltx()
    aspect = p.get("aspect") or "9:16"
    w, h = (832, 480) if aspect == "16:9" else (640, 640) if aspect == "1:1" else (480, 832)
    seconds = 10 if (p.get("seconds") or 5) > 7.5 else 5
    frames = 8 * round(seconds * 24 / 8) + 1
    steps = 30

    def cb(pipe, i, t, kw):
        job["progress"] = (i + 1) / steps
        return kw

    args = dict(prompt=p["prompt"], negative_prompt="worst quality, inconsistent motion, blurry, jittery, distorted, watermark, text",
                width=w, height=h, num_frames=frames, num_inference_steps=steps, callback_on_step_end=cb)
    if p.get("image"):
        img = Image.open(io.BytesIO(data_url_bytes(p["image"]))).convert("RGB").resize((w, h))
        out = i2v(image=img, **args).frames[0]
    else:
        out = t2v(**args).frames[0]
    path = os.path.join(OUT, job["id"] + ".mp4")
    export_to_video(out, path, fps=24)
    return path, "video/mp4"


def run_transcribe(job, p):
    path = os.path.join(OUT, job["id"] + ".wav")
    with open(path, "wb") as f:
        f.write(data_url_bytes(p["audio"]))
    segments, _ = whisper().transcribe(path, word_timestamps=True, vad_filter=True)
    words = []
    for s in segments:
        for w in (s.words or []):
            words.append({"text": w.word.strip(), "start": float(w.start), "end": float(w.end)})
    out = os.path.join(OUT, job["id"] + ".json")
    with open(out, "w") as f:
        json.dump({"words": words}, f)
    return out, "application/json"


def run_music(job, p):
    proc, model = musicgen()
    seconds = max(5, min(60, int(p.get("seconds") or 30)))
    inputs = proc(text=[p["prompt"]], padding=True, return_tensors="pt").to(model.device)
    audio = model.generate(**inputs, max_new_tokens=int(seconds * 50))
    path = os.path.join(OUT, job["id"] + ".wav")
    save_wav(path, audio[0, 0].cpu().numpy(), model.config.audio_encoder.sampling_rate)
    return path, "audio/wav"


def run_tts(job, p):
    text = p["text"]
    spanish = bool(re.search(r"[ñáéíóú¿¡]|\b(el|la|que|de|y|en|los|las|un|una|es|por|con|pero|mi|tu)\b", text.lower()))
    tok, model = tts("spa" if spanish else "eng")
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        wav = model(**inputs).waveform
    path = os.path.join(OUT, job["id"] + ".wav")
    save_wav(path, wav[0].cpu().numpy(), model.config.sampling_rate)
    return path, "audio/wav"


RUNNERS = {"video": run_video, "transcribe": run_transcribe, "music": run_music, "tts": run_tts}


def worker():
    while True:
        jid = Q.get()
        job = JOBS[jid]
        job["status"] = "running"
        try:
            job["file"], job["mime"] = RUNNERS[job["kind"]](job, job.pop("payload"))
            job["status"] = "done"
        except Exception as e:  # el error se le muestra a la app
            traceback.print_exc()
            job["status"] = "error"
            job["error"] = str(e)[:300]
        finally:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()


threading.Thread(target=worker, daemon=True).start()


def enqueue(kind, payload):
    jid = uuid.uuid4().hex[:12]
    JOBS[jid] = {"id": jid, "kind": kind, "status": "queued", "payload": payload, "progress": 0, "created": time.time()}
    Q.put(jid)
    return {"id": jid}


class VideoReq(BaseModel):
    prompt: str
    image: str | None = None
    seconds: float = 5
    aspect: str = "9:16"


class AudioReq(BaseModel):
    audio: str


class MusicReq(BaseModel):
    prompt: str
    seconds: float = 30


class TTSReq(BaseModel):
    text: str


@app.get("/")
def health():
    return {"ok": True, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "queue": Q.qsize()}


@app.post("/video")
def video(r: VideoReq):
    return enqueue("video", r.model_dump())


@app.post("/transcribe")
def transcribe(r: AudioReq):
    return enqueue("transcribe", r.model_dump())


@app.post("/music")
def music(r: MusicReq):
    return enqueue("music", r.model_dump())


@app.post("/tts")
def speak(r: TTSReq):
    return enqueue("tts", r.model_dump())


@app.get("/job/{jid}")
def job(jid: str):
    j = JOBS.get(jid)
    if not j:
        raise HTTPException(404, "no existe")
    ahead = sum(1 for o in JOBS.values() if o["status"] == "queued" and o["created"] < j["created"])
    return {"status": j["status"], "progress": j.get("progress", 0), "error": j.get("error"), "position": ahead + 1 if j["status"] == "queued" else 0}


@app.get("/file/{jid}")
def file(jid: str):
    j = JOBS.get(jid)
    if not j or j["status"] != "done":
        raise HTTPException(404, "no está listo")
    if j["mime"] == "application/json":
        with open(j["file"]) as f:
            return JSONResponse(json.load(f))
    return FileResponse(j["file"], media_type=j["mime"])


In [ ]:
# 3) Arrancar el servidor y crear tu link público
import sys, threading, subprocess, re, time
sys.path.insert(0, '/content')
import uvicorn, mimente_server
threading.Thread(target=lambda: uvicorn.run(mimente_server.app, host='0.0.0.0', port=8000, log_level='warning'), daemon=True).start()
time.sleep(3)
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
from IPython.display import display, HTML
display(HTML(f'''<div style="font:16px sans-serif;padding:16px;border:2px solid #FF4D2E;border-radius:12px">
<b>✅ Tu motor gratis está listo.</b><br>Copia este link y pégalo en la app (Ajustes → Link de tu Colab):<br>
<input value="{url}" style="width:100%;font-size:18px;padding:8px;margin-top:8px" onclick="this.select()" readonly></div>'''))

In [ ]:
# 4) (Opcional) Mantenerlo despierto y ver la cola
import requests, time
while True:
    try: print(time.strftime('%H:%M'), requests.get('http://localhost:8000/').json())
    except Exception as e: print('…', e)
    time.sleep(60)